# 11.5 Time Series Preprocessing

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 11: Time Series Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-11-time-series-machine-learning/11.05_time-series-preprocessing.ipynb)

Companion code for Section 11.5 of the book. Run the cells in order from top to bottom.

**Example 11.5.1** · bash

```bash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
import warnings; warnings.filterwarnings('ignore')

def stationarity_tests(series, name='Series'):
    '''Run ADF and KPSS tests and interpret results.'''
    print(f'\n=== Stationarity Tests: {name} ===')

    # ADF Test
    adf_result = adfuller(series.dropna(), autolag='AIC')
    adf_stat, adf_p = adf_result[0], adf_result[1]
    adf_conclusion = 'STATIONARY' if adf_p < 0.05 else 'NON-STATIONARY'
    print(f'ADF Test:  stat={adf_stat:.4f}  p={adf_p:.6f}  → {adf_conclusion}')
    print(f'  Critical values: 1%={adf_result[4]["1%"]:.4f}  5%={adf_result[4]["5%"]:.4f}')

    # KPSS Test
    kpss_result = kpss(series.dropna(), regression='c', nlags='auto')
    kpss_stat, kpss_p = kpss_result[0], kpss_result[1]
    kpss_conclusion = 'STATIONARY' if kpss_p > 0.05 else 'NON-STATIONARY'
    print(f'KPSS Test: stat={kpss_stat:.4f}  p={kpss_p:.6f}  → {kpss_conclusion}')

    # Combined conclusion
    if adf_conclusion == 'STATIONARY' and kpss_conclusion == 'STATIONARY':
        print('Combined: STATIONARY ✓')
    elif adf_conclusion == 'NON-STATIONARY' and kpss_conclusion == 'NON-STATIONARY':
        print('Combined: NON-STATIONARY - apply differencing')
    else:
        print('Combined: INCONCLUSIVE - inspect visually')
    return adf_p, kpss_p


# ── Test on various series ────────────────────────────────────────
np.random.seed(42)
n = 500

# Non-stationary: random walk
y_rw = pd.Series(np.cumsum(np.random.normal(0, 1, n)))
# Non-stationary: trend
y_trend = pd.Series(np.arange(n)*0.5 + np.random.normal(0, 3, n))
# Stationary: AR(1)
y_ar1 = pd.Series(np.array([0.8*y + np.random.normal(0, 1) for y in
                               np.concatenate([[0], np.zeros(n-1)])]))
for i in range(1, n): y_ar1.iloc[i] = 0.7*y_ar1.iloc[i-1] + np.random.normal(0,1)

stationarity_tests(y_rw,   'Random Walk')
stationarity_tests(y_trend,'Trend Series')
stationarity_tests(y_ar1,  'AR(1) - stationary')

# ── Differencing to achieve stationarity ─────────────────────────
print('\n=== Differencing to Achieve Stationarity ===')
for d in [1, 2]:
    y_diff = y_rw.diff(d).dropna()
    adf_p, _ = stationarity_tests(y_diff, f'Random Walk diff(d={d})')
    if adf_p < 0.05:
        print(f'→ d={d} differencing is sufficient!')
        break

# ── Variance stabilisation ────────────────────────────────────────
print('\n=== Variance Stabilisation ===')

# Series with increasing variance (multiplicative seasonality)
t_arr = np.arange(1, n+1)
y_hetero = pd.Series(t_arr * (1 + 0.3*np.sin(2*np.pi*t_arr/52)) +
                      np.random.normal(0, t_arr*0.05, n))

print(f'Original variance (rolling std range): {y_hetero.rolling(50).std().dropna().describe()["min"]:.2f} to
{y_hetero.rolling(50).std().dropna().describe()["max"]:.2f}')

# Log transform
y_log = np.log1p(y_hetero.clip(lower=0.01))
print(f'Log-transformed range:                 {y_log.rolling(50).std().dropna().describe()["min"]:.4f} to
{y_log.rolling(50).std().dropna().describe()["max"]:.4f}')

# Box-Cox transform
from scipy.stats import boxcox
y_bc, lam = boxcox(y_hetero.clip(lower=0.01))
y_bc = pd.Series(y_bc)
print(f'Box-Cox (λ={lam:.3f}) range:              {y_bc.rolling(50).std().dropna().describe()["min"]:.4f} to
{y_bc.rolling(50).std().dropna().describe()["max"]:.4f}')
```